# Study 954 — High Yield in Disguise 🎭

**Is a high-yield bond fund just equity and Treasuries in a costume — and if not, does the
difference pay?**

The trade-desk aphorism says a junk bond is a senior claim on a leveraged company: equity
risk on the way down, capped on the way up. If that is the whole story, a high-yield fund
should be reproducible as a simple blend of **SPY** (equity) and **IEF** (Treasury
duration) — and you would be paying a bond fund's fee for a mixture you could hold yourself.

We fit that blend to **HYG** out of sample — a trailing 252-day constrained regression,
the weight frozen at each month-end and applied to the *following* month — and race the two
**excess-of-cash** (BIL), over 2008-06-02 → 2026-06-30 (4,548 days) of daily
**total-return** closes, 2 bps one-way on the blend's rebalance.

*Every real-tape number below is the frozen headline from `docs/results.md`
(Fingerprint `c08a8d88f0f9`, as-of 2026-06-30). The live cells at the end run an offline
**synthetic** control and are labelled as such.*


## 1. What the fitted recipe turned out to be

Solve for the equity share `w` that best explains HYG's daily moves, refit it every month, and never let the fit see the future. Over eighteen years the answer barely wandered.

In [1]:
R = dict(w_mean=0.45, w_min=0.269, w_max=0.631, turnover=0.34, short_max=0.0)
print('fitted recipe for a high-yield fund: %.0f%% equity + %.0f%% Treasuries'
      % (R['w_mean']*100, (1-R['w_mean'])*100))
print('the equity share ranged from %.2f to %.2f over 18 years' % (R['w_min'], R['w_max']))
print('never short, never levered (max short notional %.3f); %.2f x NAV traded per year'
      % (R['short_max'], R['turnover']))

fitted recipe for a high-yield fund: 45% equity + 55% Treasuries
the equity share ranged from 0.27 to 0.63 over 18 years
never short, never levered (max short notional 0.000); 0.34 x NAV traded per year


So high yield is **not** levered equity. It is roughly **45% equity and 55% Treasuries** — *de*-levered equity, with a big bond sleeve attached.

> 🔬 **For the quants:** the weight comes from regressing `r_HY − r_IEF` on `r_SPY − r_IEF`. Subtracting the duration leg from both sides imposes *weights sum to one* exactly, so the slope is the equity share of a fully funded blend — no cash is created or destroyed and no leverage sneaks in.

## 2. The costume does not fit

If high yield really were the blend, the blend would explain almost all of it. It explains about half — and stretching the return horizon (which would flatter a fund whose bonds are marked stale) barely helps.

In [2]:
R = dict(r2_d=0.468, r2_w=0.565, r2_m=0.517, r2_q=0.591, te_d=8.15, te_q=6.26)
for label, r2, in [('daily', R['r2_d']), ('weekly', R['r2_w']),
                   ('monthly', R['r2_m']), ('quarterly', R['r2_q'])]:
    print('%-10s the blend explains %.0f%% of high yield' % (label, r2*100))
print()
print('left over: %.1f%% a year of tracking error (daily), %.1f%% (quarterly)'
      % (R['te_d'], R['te_q']))

daily      the blend explains 47% of high yield
weekly     the blend explains 56% of high yield
monthly    the blend explains 52% of high yield
quarterly  the blend explains 59% of high yield

left over: 8.2% a year of tracking error (daily), 6.3% (quarterly)


Roughly **7 percentage points a year** of high yield's movement is something neither the stock market nor the Treasury market does. That is credit risk — real, distinct, and yours to be paid for. **The costume story fails.**

## 3. But the distinct risk paid nothing

So the fund is not a copy. The next question is the only one that matters to an owner: for the same amount of risk taken, who ended up with more money?

In [3]:
R = dict(hy_sharpe=0.395, rp_sharpe=0.731, hy_cagr=3.85, rp_cagr=6.34,
         hy_lived=5.17, rp_lived=7.68, cash_lived=1.26,
         hy_vol=11.12, rp_vol=8.96, hy_dd=-32.4, rp_dd=-20.7, gap=-0.336, t_gap=-2.07,
         gap_pp_per_yr=3.7)
print('what the statement showed (total return, cash made %+.2f%% a year):'
      % R['cash_lived'])
print('  HYG, held 18 years : %+.2f%% a year, %.1f%% volatility, worst loss %.1f%%'
      % (R['hy_lived'], R['hy_vol'], R['hy_dd']))
print('  the 45/55 blend    : %+.2f%% a year, %.1f%% volatility, worst loss %.1f%%'
      % (R['rp_lived'], R['rp_vol'], R['rp_dd']))
print()
print('the same race after subtracting cash (what the Sharpe compares):')
print('  above cash: %+.2f%% a year for the fund, %+.2f%% for the blend'
      % (R['hy_cagr'], R['rp_cagr']))
print('  return per unit of risk: %.3f for the fund, %.3f for the blend'
      % (R['hy_sharpe'], R['rp_sharpe']))
print('gap %+.3f  (statistical t = %+.2f)  ~%.1f pp/yr at matched volatility'
      % (R['gap'], R['t_gap'], R['gap_pp_per_yr']))

what the statement showed (total return, cash made +1.26% a year):
  HYG, held 18 years : +5.17% a year, 11.1% volatility, worst loss -32.4%
  the 45/55 blend    : +7.68% a year, 9.0% volatility, worst loss -20.7%

the same race after subtracting cash (what the Sharpe compares):
  above cash: +3.85% a year for the fund, +6.34% for the blend
  return per unit of risk: 0.395 for the fund, 0.731 for the blend
gap -0.336  (statistical t = -2.07)  ~3.7 pp/yr at matched volatility


The homemade blend won on **every** count over these eighteen years: more return, less volatility, a shallower worst loss. Part of that is simply the fee — HYG charges **0.49%** a year against **0.125%** for the blend, a **0.36 pp** head start — but the fee is only about a sixth of the gap. The rest is the credit risk failing to pay.

> 🔬 **For the quants:** the *t* on that gap is **-2.07** and the block-bootstrap CI is [-0.81, -0.04] — clear of zero, but barely. The direction is unanimous across four estimation windows, four Treasury maturities, both eras and all three funds; the *significance* is not — swap the Treasury leg for SHY (-1.93) or TLT (-1.92) and the headline drops back under the bar. The magnitude is one good year away from being unremarkable. That is why the Signal stamp is Mixed and not Real.

## 4. Except in 2022 — and that is the catch

Three crises, three verdicts. Two of them are lopsided wins for the blend, because when *credit* is what breaks, the blend's Treasury sleeve rallies while high-yield spreads blow out. The third goes the other way.

In [4]:
R = dict(c08_dd_hy=-32.4, c08_dd_rp=-20.7, c20_dd_hy=-22.0, c20_dd_rp=-13.1,
         c20_ret_hy=-6.9, c20_ret_rp=1.4, c22_dd_hy=-15.5, c22_dd_rp=-18.9,
         c22_ret_hy=-11.0, c22_ret_rp=-15.7)
print('2008 credit crisis : fund %.1f%% drawdown vs blend %.1f%%'
      % (R['c08_dd_hy'], R['c08_dd_rp']))
print('2020 Covid crash   : fund %.1f%% drawdown vs blend %.1f%%'
      % (R['c20_dd_hy'], R['c20_dd_rp']))
print('   over that quarter the fund lost %.1f%%; the blend made %+.1f%%'
      % (R['c20_ret_hy'], R['c20_ret_rp']))
print()
print('2022 rate shock    : fund %.1f%% drawdown vs blend %.1f%%  <- the blend LOSES'
      % (R['c22_dd_hy'], R['c22_dd_rp']))
print('   over that year the fund lost %.1f%%; the blend lost %.1f%%'
      % (R['c22_ret_hy'], R['c22_ret_rp']))

2008 credit crisis : fund -32.4% drawdown vs blend -20.7%
2020 Covid crash   : fund -22.0% drawdown vs blend -13.1%
   over that quarter the fund lost -6.9%; the blend made +1.4%

2022 rate shock    : fund -15.5% drawdown vs blend -18.9%  <- the blend LOSES
   over that year the fund lost -11.0%; the blend lost -15.7%


2022 was not a credit event — it was a **pure interest-rate** event, and the blend carries far more interest-rate risk than the fund does (high-yield bonds are shorter and pay more coupon). Swapping the fund for the blend is not a free lunch: it is **trading credit risk for duration risk**. Over these eighteen years that trade paid; in the one year rates were the story, it cost 4.7 pp.

## 5. Is the harness honest? (a live, offline synthetic check)

Everything above is a *frozen* real-tape number. The two cells below are **synthetic** — a made-up world where we control the answer — run live to prove the machinery finds an effect when one is planted and stays quiet when it is not.

The synthetic fund is genuinely 45% equity + 55% duration *plus* a credit shock of fixed size. In world A the shock is paid nothing; in world B — the null — it is paid exactly enough to keep the fund level with its blend.

In [5]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
from hy_replication import data, strategy as st

for label, ss in [('A  shock paid nothing ', 1.0), ('B  shock fairly paid  ', 0.0)]:
    prices, truth = data.synthetic_panel(signal_strength=ss, seed=954)
    d = st.synthetic_detect(prices)
    print('%s: recipe found %.3f (planted %.2f) | blend ahead by %+.3f'
          % (label, d['w_mean'], truth['w_true'], -d['excess_sharpe_gap']))
print()
print('SYNTHETIC, not the real tape: the recipe is recovered either way, but the')
print('blend only WINS when the extra risk is genuinely unpaid. World B is one')
print('draw of a null whose spread is 0.13, so its small lead is noise: across')
print('8 seeds world B averages -0.05 and never fires, world A averages -0.44.')

A  shock paid nothing : recipe found 0.456 (planted 0.45) | blend ahead by +0.511


B  shock fairly paid  : recipe found 0.456 (planted 0.45) | blend ahead by +0.126

SYNTHETIC, not the real tape: the recipe is recovered either way, but the
blend only WINS when the extra risk is genuinely unpaid. World B is one
draw of a null whose spread is 0.13, so its small lead is noise: across
8 seeds world B averages -0.05 and never fires, world A averages -0.44.


## Verdict

- **Signal — Mixed.** Half the folklore is simply wrong: high yield is *not* a repackaged equity position — a held-out 45/55 blend reproduces under half of it (R² 0.47) and leaves ~7 pp/yr of genuinely different risk. The other half — that you are not paid for that difference — is true in **every** cut we took, but only just: *t* = -2.07, bootstrap CI [-0.81, -0.04], with JNK, USHY and the first era all short of the bar.
- **Tradability — Fragile.** The swap is cheap, low-turnover and gained +2.5 pp/yr of lived CAGR with a 11.7 pp shallower worst loss — but it is a substitution, not an edge, and it hands you a different risk. In the one year that risk showed up (2022) the blend lost 4.7 pp more than the fund it replaced.